In [ ]:
!pip install numpy scikit-learn tensorflow

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
import matplotlib.pyplot as plt
import seaborn as sns
import time

print("=" * 80)
print("BEST MODEL TRAINING & COMPREHENSIVE ANALYSIS - 12 CLASSES")
print("Best Config: CBAM + 3 Residual Blocks")
print("=" * 80)

gpus = tf.config.list_physical_devices('GPU')
print(f"✓ GPU: {gpus}")
print("=" * 80)


# ============================================================================
# QUANTUM LAYERS
# ============================================================================

class MultiScaleQuantumConv(layers.Layer):
    def __init__(self, filters, n_qubits=4, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.n_qubits = n_qubits

    def build(self, input_shape):
        actual_channels = input_shape[-1]

        self.q_encoding_fine = self.add_weight(
            shape=(actual_channels, self.n_qubits),
            initializer='glorot_uniform',
            trainable=True,
            name='q_encode_fine'
        )

        self.q_encoding_coarse = self.add_weight(
            shape=(actual_channels, self.n_qubits),
            initializer='glorot_uniform',
            trainable=True,
            name='q_encode_coarse'
        )

        self.theta_layer1 = self.add_weight(
            shape=(self.n_qubits,),
            initializer=keras.initializers.RandomUniform(-np.pi, np.pi),
            trainable=True,
            name='theta_1'
        )

        self.theta_layer2 = self.add_weight(
            shape=(self.n_qubits,),
            initializer=keras.initializers.RandomUniform(-np.pi, np.pi),
            trainable=True,
            name='theta_2'
        )

        self.entangle_strength = self.add_weight(
            shape=(self.n_qubits - 1,),
            initializer='ones',
            trainable=True,
            name='entangle'
        )

        self.q_proj = self.add_weight(
            shape=(self.n_qubits * 4, self.filters),
            initializer='glorot_uniform',
            trainable=True,
            name='q_proj'
        )

    def quantum_rotation(self, state_0, state_1, theta):
        cos_t = tf.cos(theta / 2.0)
        sin_t = tf.sin(theta / 2.0)
        new_s0 = cos_t * state_0 - sin_t * state_1
        new_s1 = sin_t * state_0 + cos_t * state_1
        return new_s0, new_s1

    def quantum_entanglement(self, state_0, state_1):
        state_0_list = tf.unstack(state_0, axis=-1)
        state_1_list = tf.unstack(state_1, axis=-1)

        for i in range(self.n_qubits - 1):
            strength = tf.nn.sigmoid(self.entangle_strength[i])
            control = tf.abs(state_1_list[i])
            target_flip = strength * control * 0.4

            state_0_target = state_0_list[i+1]
            state_1_target = state_1_list[i+1]

            state_0_list[i+1] = state_0_target * (1 - target_flip) + state_1_target * target_flip
            state_1_list[i+1] = state_1_target * (1 - target_flip) + state_0_target * target_flip

        state_0 = tf.stack(state_0_list, axis=-1)
        state_1 = tf.stack(state_1_list, axis=-1)
        return state_0, state_1

    def call(self, inputs):
        batch_size = tf.shape(inputs)[0]

        fine_pool = tf.reduce_mean(inputs, axis=[1, 2])
        coarse_pool = tf.nn.avg_pool2d(inputs, ksize=2, strides=2, padding='VALID')
        coarse_pool = tf.reduce_mean(coarse_pool, axis=[1, 2])

        angles_fine = tf.tanh(tf.matmul(fine_pool, self.q_encoding_fine))
        state_0_fine = tf.cos(angles_fine / 2.0)
        state_1_fine = tf.sin(angles_fine / 2.0)

        angles_coarse = tf.tanh(tf.matmul(coarse_pool, self.q_encoding_coarse))
        state_0_coarse = tf.cos(angles_coarse / 2.0)
        state_1_coarse = tf.sin(angles_coarse / 2.0)

        state_0_fine, state_1_fine = self.quantum_rotation(
            state_0_fine, state_1_fine, self.theta_layer1
        )
        state_0_coarse, state_1_coarse = self.quantum_rotation(
            state_0_coarse, state_1_coarse, self.theta_layer1
        )

        state_0_fine, state_1_fine = self.quantum_entanglement(state_0_fine, state_1_fine)
        state_0_coarse, state_1_coarse = self.quantum_entanglement(state_0_coarse, state_1_coarse)

        state_0_fine, state_1_fine = self.quantum_rotation(
            state_0_fine, state_1_fine, self.theta_layer2
        )
        state_0_coarse, state_1_coarse = self.quantum_rotation(
            state_0_coarse, state_1_coarse, self.theta_layer2
        )

        quantum_state = tf.concat([
            state_0_fine, state_1_fine,
            state_0_coarse, state_1_coarse
        ], axis=-1)

        q_features = tf.matmul(quantum_state, self.q_proj)
        q_features = tf.nn.swish(q_features)

        height, width = tf.shape(inputs)[1], tf.shape(inputs)[2]
        q_features = tf.expand_dims(tf.expand_dims(q_features, 1), 1)
        q_features = tf.tile(q_features, [1, height, width, 1])

        return q_features

    def get_config(self):
        config = super().get_config()
        config.update({"filters": self.filters, "n_qubits": self.n_qubits})
        return config

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[1], input_shape[2], self.filters)


class QuantumFeatureFusion(layers.Layer):
    def __init__(self, n_qubits=5, **kwargs):
        super().__init__(**kwargs)
        self.n_qubits = n_qubits

    def build(self, input_shape):
        self.feature_dim = input_shape[-1]

        self.encoding_A = self.add_weight(
            shape=(self.feature_dim, self.n_qubits),
            initializer='glorot_uniform',
            trainable=True,
            name='encode_A'
        )

        self.encoding_B = self.add_weight(
            shape=(self.feature_dim, self.n_qubits),
            initializer='glorot_uniform',
            trainable=True,
            name='encode_B'
        )

        self.phase_shift = self.add_weight(
            shape=(self.n_qubits,),
            initializer=keras.initializers.RandomUniform(-np.pi, np.pi),
            trainable=True,
            name='phase'
        )

        self.measurement = self.add_weight(
            shape=(self.n_qubits * 2, self.feature_dim),
            initializer='glorot_uniform',
            trainable=True,
            name='measure'
        )

    def call(self, inputs):
        angles_A = tf.tanh(tf.matmul(inputs, self.encoding_A))
        angles_B = tf.tanh(tf.matmul(inputs, self.encoding_B))

        state_A_0 = tf.cos(angles_A / 2.0)
        state_A_1 = tf.sin(angles_A / 2.0)

        state_B_0 = tf.cos(angles_B / 2.0)
        state_B_1 = tf.sin(angles_B / 2.0)

        cos_phase = tf.cos(self.phase_shift / 2.0)
        sin_phase = tf.sin(self.phase_shift / 2.0)

        fused_0 = (state_A_0 + state_B_0 * cos_phase) / tf.sqrt(2.0)
        fused_1 = (state_A_1 + state_B_1 * sin_phase) / tf.sqrt(2.0)

        norm = tf.sqrt(tf.square(fused_0) + tf.square(fused_1) + 1e-8)
        fused_0 = fused_0 / norm
        fused_1 = fused_1 / norm

        quantum_features = tf.concat([fused_0, fused_1], axis=-1)
        output = tf.matmul(quantum_features, self.measurement)

        return output + inputs

    def get_config(self):
        config = super().get_config()
        config.update({"n_qubits": self.n_qubits})
        return config

    def compute_output_shape(self, input_shape):
        return input_shape


# ============================================================================
# ATTENTION MECHANISMS
# ============================================================================

class ChannelAttention(layers.Layer):
    def __init__(self, reduction_ratio=16, **kwargs):
        super().__init__(**kwargs)
        self.reduction_ratio = reduction_ratio

    def build(self, input_shape):
        channels = input_shape[-1]
        self.fc1 = layers.Dense(channels // self.reduction_ratio, activation='relu')
        self.fc2 = layers.Dense(channels, activation='sigmoid')

    def call(self, inputs):
        x = tf.reduce_mean(inputs, axis=[1, 2], keepdims=True)
        x = self.fc1(x)
        x = self.fc2(x)
        return inputs * x

    def get_config(self):
        config = super().get_config()
        config.update({"reduction_ratio": self.reduction_ratio})
        return config


class SpatialAttention(layers.Layer):
    def __init__(self, kernel_size=7, **kwargs):
        super().__init__(**kwargs)
        self.kernel_size = kernel_size

    def build(self, input_shape):
        self.conv = layers.Conv2D(1, self.kernel_size, padding='same', activation='sigmoid')

    def call(self, inputs):
        max_pool = tf.reduce_max(inputs, axis=-1, keepdims=True)
        avg_pool = tf.reduce_mean(inputs, axis=-1, keepdims=True)
        x = tf.concat([max_pool, avg_pool], axis=-1)
        x = self.conv(x)
        return inputs * x

    def get_config(self):
        config = super().get_config()
        config.update({"kernel_size": self.kernel_size})
        return config


class CBAM(layers.Layer):
    def __init__(self, reduction_ratio=16, kernel_size=7, **kwargs):
        super().__init__(**kwargs)
        self.reduction_ratio = reduction_ratio
        self.kernel_size = kernel_size

    def build(self, input_shape):
        self.channel_attention = ChannelAttention(self.reduction_ratio)
        self.spatial_attention = SpatialAttention(self.kernel_size)

    def call(self, inputs):
        x = self.channel_attention(inputs)
        x = self.spatial_attention(x)
        return x

    def get_config(self):
        config = super().get_config()
        config.update({
            "reduction_ratio": self.reduction_ratio,
            "kernel_size": self.kernel_size
        })
        return config


# ============================================================================
# BEST MODEL - CBAM + 3 RESIDUAL BLOCKS (12 CLASSES)
# ============================================================================

def build_best_model(input_shape=(40, 24, 1), num_classes=12):
    """
    Best configuration adapted for 12-class classification:
    - 3 residual blocks
    - CBAM attention in each residual block
    - Quantum fusion
    - Branch attention
    """
    inputs = layers.Input(shape=input_shape, name='input')

    # Initial feature extraction
    x = layers.Conv2D(48, (3, 3), padding='same', kernel_initializer='he_normal')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.2)(x)

    # Stage 1
    classical_path = layers.SeparableConv2D(48, 3, padding='same',
                                            depthwise_initializer='he_normal',
                                            pointwise_initializer='he_normal')(x)
    classical_path = layers.BatchNormalization()(classical_path)
    classical_path = layers.Activation('relu')(classical_path)
    classical_path = layers.Dropout(0.25)(classical_path)

    classical_path = layers.SeparableConv2D(48, 3, padding='same',
                                            depthwise_initializer='he_normal',
                                            pointwise_initializer='he_normal')(classical_path)
    classical_path = layers.BatchNormalization()(classical_path)
    classical_path = layers.Activation('relu')(classical_path)

    x = classical_path
    x = layers.MaxPooling2D((2, 2))(x)

    # Stage 2
    classical_path = layers.SeparableConv2D(72, 3, padding='same',
                                            depthwise_initializer='he_normal',
                                            pointwise_initializer='he_normal')(x)
    classical_path = layers.BatchNormalization()(classical_path)
    classical_path = layers.Activation('relu')(classical_path)
    classical_path = layers.Dropout(0.3)(classical_path)

    classical_path = layers.SeparableConv2D(72, 3, padding='same',
                                            depthwise_initializer='he_normal',
                                            pointwise_initializer='he_normal')(classical_path)
    classical_path = layers.BatchNormalization()(classical_path)
    classical_path = layers.Activation('relu')(classical_path)

    x = classical_path
    x = layers.Conv2D(96, 1, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)

    # Stage 3: 3 RESIDUAL BLOCKS WITH CBAM ATTENTION
    for block_idx in range(3):
        shortcut = x

        x = layers.SeparableConv2D(96, 3, padding='same',
                                  depthwise_initializer='he_normal',
                                  pointwise_initializer='he_normal')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Dropout(0.35)(x)

        x = layers.SeparableConv2D(96, 3, padding='same',
                                  depthwise_initializer='he_normal',
                                  pointwise_initializer='he_normal')(x)
        x = layers.BatchNormalization()(x)

        # CBAM Attention
        x = CBAM(reduction_ratio=8, kernel_size=7)(x)

        # Residual connection
        x = layers.Add()([shortcut, x])
        x = layers.Activation('relu')(x)

    x = layers.Dropout(0.4)(x)
    x = layers.GlobalAveragePooling2D()(x)

    # Quantum Fusion
    x = QuantumFeatureFusion(n_qubits=5, name='quantum_fusion')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('swish')(x)
    x = layers.Dropout(0.45)(x)

    # Classification Head with Branch Attention
    branch1 = layers.Dense(128, kernel_initializer='he_normal',
                          kernel_regularizer=keras.regularizers.l2(1e-3))(x)
    branch1 = layers.BatchNormalization()(branch1)
    branch1 = layers.Activation('swish')(branch1)
    branch1 = layers.Dropout(0.5)(branch1)

    branch2 = layers.Dense(96, kernel_initializer='he_normal',
                          kernel_regularizer=keras.regularizers.l2(1e-3))(x)
    branch2 = layers.BatchNormalization()(branch2)
    branch2 = layers.Activation('swish')(branch2)
    branch2 = layers.Dropout(0.5)(branch2)

    branch2 = layers.Dense(64, kernel_initializer='he_normal',
                          kernel_regularizer=keras.regularizers.l2(1e-3))(branch2)
    branch2 = layers.BatchNormalization()(branch2)
    branch2 = layers.Activation('swish')(branch2)
    branch2 = layers.Dropout(0.5)(branch2)

    merged = layers.Concatenate()([branch1, branch2])

    # Branch Attention
    attention = layers.Dense(192, activation='sigmoid')(merged)
    x = layers.Multiply()([merged, attention])

    x = layers.Dense(96, kernel_initializer='he_normal',
                     kernel_regularizer=keras.regularizers.l2(1e-3))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('swish')(x)
    x = layers.Dropout(0.5)(x)

    # 12-class output
    outputs = layers.Dense(num_classes, activation='softmax',
                          kernel_regularizer=keras.regularizers.l2(1e-3))(x)

    model = models.Model(inputs=inputs, outputs=outputs, name='Best_CBAM_3Residual_12Class')
    return model


# ============================================================================
# ENHANCED VISUALIZATION FUNCTIONS
# ============================================================================

def plot_training_curves(history, save_path='training_curves.png'):
    """Plot training and validation accuracy/loss curves"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    epochs = range(1, len(history.history['accuracy']) + 1)

    # Accuracy
    ax1.plot(epochs, history.history['accuracy'], 'b-', label='Training Accuracy', linewidth=2)
    ax1.plot(epochs, history.history['val_accuracy'], 'r-', label='Validation Accuracy', linewidth=2)
    ax1.set_title('Model Accuracy', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Accuracy', fontsize=12)
    ax1.legend(fontsize=10)
    ax1.grid(True, alpha=0.3)

    # Loss
    ax2.plot(epochs, history.history['loss'], 'b-', label='Training Loss', linewidth=2)
    ax2.plot(epochs, history.history['val_loss'], 'r-', label='Validation Loss', linewidth=2)
    ax2.set_title('Model Loss', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Loss', fontsize=12)
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Saved training curves: {save_path}")
    plt.close()


def plot_confusion_matrix(y_true, y_pred, class_names, save_path='confusion_matrix.png'):
    """Plot enhanced confusion matrix with custom styling"""
    cm = confusion_matrix(y_true, y_pred)

    # Create figure
    fig, ax = plt.subplots(figsize=(12, 8))

    # Plot heatmap
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
                linewidths=0, annot_kws={"size": 14}, ax=ax)

    # Get color bar and increase its font size
    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(labelsize=14)

    # Set thick borders around the figure
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(3)

    # Set custom ticks
    ax.set_xticks(np.arange(12) + 0.5)
    ax.set_yticks(np.arange(12) + 0.5)

    # Set labels with rotation
    ax.set_xticklabels(class_names, fontsize=18, rotation=45, ha='right')
    ax.set_yticklabels(class_names, fontsize=16, rotation=45)

    # Add labels
    plt.xlabel('Predicted Labels', fontsize=20)
    plt.ylabel('True Labels', fontsize=20)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Saved confusion matrix: {save_path}")
    plt.close()

    return cm


def plot_normalized_confusion_matrix(y_true, y_pred, class_names, save_path='confusion_matrix_normalized.png'):
    """Plot normalized confusion matrix with custom styling"""
    cm = confusion_matrix(y_true, y_pred)
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    # Create figure
    fig, ax = plt.subplots(figsize=(12, 8))

    # Plot normalized heatmap
    sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues', cbar=True,
                linewidths=0, annot_kws={"size": 14}, ax=ax, vmin=0, vmax=1)

    # Get color bar and increase its font size
    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(labelsize=14)

    # Set thick borders around the figure
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(3)

    # Set custom ticks
    ax.set_xticks(np.arange(12) + 0.5)
    ax.set_yticks(np.arange(12) + 0.5)

    # Set labels with rotation
    ax.set_xticklabels(class_names, fontsize=18, rotation=45, ha='right')
    ax.set_yticklabels(class_names, fontsize=16, rotation=45)

    # Add labels
    plt.xlabel('Predicted Labels', fontsize=20)
    plt.ylabel('True Labels', fontsize=20)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Saved normalized confusion matrix: {save_path}")
    plt.close()


def plot_per_class_metrics(precision, recall, f1, class_names, save_path='per_class_metrics.png'):
    """Plot per-class precision, recall, and F1-score"""
    x = np.arange(len(class_names))
    width = 0.25

    fig, ax = plt.subplots(figsize=(16, 6))

    bars1 = ax.bar(x - width, precision, width, label='Precision', alpha=0.8, color='#2ecc71')
    bars2 = ax.bar(x, recall, width, label='Recall', alpha=0.8, color='#3498db')
    bars3 = ax.bar(x + width, f1, width, label='F1-Score', alpha=0.8, color='#e74c3c')

    ax.set_xlabel('Class', fontsize=12, fontweight='bold')
    ax.set_ylabel('Score', fontsize=12, fontweight='bold')
    ax.set_title('Per-Class Metrics', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(class_names, rotation=45, ha='right')
    ax.legend(fontsize=10)
    ax.set_ylim([0, 1.05])
    ax.grid(axis='y', alpha=0.3)

    # Add value labels on bars
    for bars in [bars1, bars2, bars3]:
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.2f}',
                   ha='center', va='bottom', fontsize=7)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Saved per-class metrics: {save_path}")
    plt.close()


def plot_roc_curves(y_true, y_pred_proba, class_names, save_path='roc_curves.png'):
    """Plot ROC curves for each class"""
    from sklearn.metrics import roc_curve, auc
    from sklearn.preprocessing import label_binarize

    # Binarize the output
    y_true_bin = label_binarize(y_true, classes=range(len(class_names)))

    fig, ax = plt.subplots(figsize=(12, 10))

    colors = plt.cm.get_cmap('tab20')(np.linspace(0, 1, len(class_names)))

    for i, (class_name, color) in enumerate(zip(class_names, colors)):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_proba[:, i])
        roc_auc = auc(fpr, tpr)

        ax.plot(fpr, tpr, color=color, lw=2,
               label=f'{class_name} (AUC = {roc_auc:.2f})')

    ax.plot([0, 1], [0, 1], 'k--', lw=3, label='Random (AUC = 0.50)')
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    plt.xticks(fontsize=22)
    plt.yticks(fontsize=22)
    ax.set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
    ax.set_ylabel('True Positive Rate', fontsize=12, fontweight='bold')
    ax.set_title('ROC Curves - 12-Class Classification', fontsize=14, fontweight='bold')
    ax.legend(loc="lower right", fontsize=16, ncol=2)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Saved ROC curves: {save_path}")
    plt.close()


# ============================================================================
# MAIN
# ============================================================================

def main():
    # Load data
    print("\nLOADING DATA...")
    X_train = np.load('X_train_n12c.npy')
    y_train = np.load('y_train_n12c.npy')
    X_val = np.load('X_val_n12c.npy')
    y_val = np.load('y_val_n12c.npy')
    X_test = np.load('X_test_n12c.npy')
    y_test = np.load('y_test_n12c.npy')

    y_train_int = np.argmax(y_train, axis=1)
    y_val_int = np.argmax(y_val, axis=1)
    y_test_int = np.argmax(y_test, axis=1)

    print(f"✓ Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

    # Class names for 12 classes - matching your custom format
    class_names = ['     Cell     ', '  Cell-Multi  ', '   Cracking   ',
                   '     Diode    ', '  Diode-Multi ', '   Hot-spot   ',
                   'Hot-spot-Multi', '  No-anomaly  ', 'Offline-Module',
                   '  Shadowing   ', '    Soiling   ', '  Vegetarian  ']

    # Class weights for 12 classes
    class_counts = np.array([np.sum(y_train_int == i) for i in range(12)])
    total_samples = class_counts.sum()
    class_weights = {}

    print("\nClass Distribution:")
    for i in range(12):
        count = class_counts[i]
        percentage = (count / total_samples) * 100
        print(f"  {class_names[i].strip()}: {count} samples ({percentage:.2f}%)")

    # Calculate balanced class weights
    for i in range(12):
        weight = (total_samples / (12 * class_counts[i])) ** 0.60
        class_weights[i] = min(weight, 10.0)

    print("\nClass Weights:")
    for i in range(12):
        print(f"  {class_names[i].strip()}: {class_weights[i]:.3f}")

    # Build model
    print("\n" + "=" * 80)
    print("BUILDING BEST MODEL (12-CLASS CLASSIFICATION)")
    print("=" * 80)

    model = build_best_model(input_shape=(40, 24, 1), num_classes=12)
    model.summary()

    total_params = model.count_params()
    print(f"\n✓ Total Parameters: {total_params:,}")

    # Compile
    optimizer = keras.optimizers.Adam(learning_rate=0.001, clipnorm=1.0)
    model.compile(
        optimizer=optimizer,
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
        metrics=['accuracy']
    )

    # Callbacks
    callbacks_list = [
        callbacks.ModelCheckpoint(
            'best_model_12class.keras',
            monitor='val_accuracy',
            save_best_only=True,
            mode='max',
            verbose=1
        ),
        callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=10,
            min_lr=1e-7,
            verbose=1
        ),
        callbacks.EarlyStopping(
            monitor='val_accuracy',
            patience=35,
            restore_best_weights=True,
            mode='max',
            min_delta=0.0005,
            verbose=1
        )
    ]

    # Train
    print("\n" + "=" * 80)
    print("TRAINING BEST MODEL")
    print("=" * 80)

    start_time = time.time()

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=120,
        batch_size=64,
        class_weight=class_weights,
        callbacks=callbacks_list,
        verbose=1
    )

    training_time = (time.time() - start_time) / 60

    # Save models
    print("\n" + "=" * 80)
    print("SAVING MODELS")
    print("=" * 80)

    model.save('best_model_12class_final.keras')
    print("✓ Saved: best_model_12class_final.keras")

    model.save('best_model_12class_final.h5')
    print("✓ Saved: best_model_12class_final.h5")

    # Evaluate
    print("\n" + "=" * 80)
    print("EVALUATION")
    print("=" * 80)

    train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
    val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

    print(f"\nTrain Acc: {train_acc*100:.2f}%  |  Loss: {train_loss:.4f}")
    print(f"Val Acc:   {val_acc*100:.2f}%  |  Loss: {val_loss:.4f}")
    print(f"Test Acc:  {test_acc*100:.2f}%  |  Loss: {test_loss:.4f}")

    # TTA
    print("\n" + "=" * 80)
    print("TEST TIME AUGMENTATION (20 iterations)")
    print("=" * 80)

    tta_preds = []
    for i in range(20):
        pred = model.predict(X_test, verbose=0)
        tta_preds.append(pred)
        if (i + 1) % 5 == 0:
            print(f"  Iteration {i+1}/20 complete")

    tta_pred_proba = np.mean(tta_preds, axis=0)
    tta_pred = np.argmax(tta_pred_proba, axis=1)
    tta_acc = np.mean(tta_pred == y_test_int)

    print(f"\n✓ TTA Accuracy: {tta_acc*100:.2f}%")

    # Get single prediction for probability analysis
    y_pred_proba = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_pred_proba, axis=1)

    # Metrics
    print("\n" + "=" * 80)
    print("DETAILED METRICS")
    print("=" * 80)

    precision, recall, f1, support = precision_recall_fscore_support(
        y_test_int, tta_pred, average=None
    )

    print("\nPer-Class Metrics:")
    print(f"{'Class':<18s} {'Precision':<12s} {'Recall':<12s} {'F1-Score':<12s} {'Support':<10s}")
    print("-" * 70)
    for i, class_name in enumerate(class_names):
        print(f"{class_name:<18s} {precision[i]:>10.4f}  {recall[i]:>10.4f}  "
              f"{f1[i]:>10.4f}  {support[i]:>8d}")

    # Average metrics
    precision_avg = np.mean(precision)
    recall_avg = np.mean(recall)
    f1_avg = np.mean(f1)

    print("\n" + "-" * 70)
    print(f"{'Average':<18s} {precision_avg:>10.4f}  {recall_avg:>10.4f}  {f1_avg:>10.4f}")

    # Classification report
    print("\n" + "=" * 80)
    print("CLASSIFICATION REPORT")
    print("=" * 80)
    print(classification_report(y_test_int, tta_pred,
                                target_names=[name.strip() for name in class_names]))

    # Visualizations
    print("\n" + "=" * 80)
    print("GENERATING VISUALIZATIONS")
    print("=" * 80)

    plot_training_curves(history, 'training_curves_12class.png')
    plot_confusion_matrix(y_test_int, tta_pred, class_names, 'confusion_matrix_12class.png')
    plot_normalized_confusion_matrix(y_test_int, tta_pred, class_names, 'confusion_matrix_normalized_12class.png')

    # Use stripped class names for metrics plot
    plot_per_class_metrics(precision, recall, f1,
                          [name.strip() for name in class_names],
                          'per_class_metrics_12class.png')
    plot_roc_curves(y_test_int, tta_pred_proba,
                   [name.strip() for name in class_names],
                   'roc_curves_12class.png')

    # Summary
    print("\n" + "=" * 80)
    print("🎉 FINAL SUMMARY")
    print("=" * 80)
    print(f"Model: CBAM + 3 Residual Blocks (12-Class Classification)")
    print(f"Parameters: {total_params:,}")
    print(f"Training Time: {training_time:.1f} minutes")
    print(f"\nAccuracy:")
    print(f"  Train: {train_acc*100:.2f}%")
    print(f"  Val:   {val_acc*100:.2f}%")
    print(f"  Test:  {test_acc*100:.2f}%")
    print(f"  TTA:   {tta_acc*100:.2f}%")
    print(f"\nMetrics (Average):")
    print(f"  Precision: {precision_avg:.4f}")
    print(f"  Recall:    {recall_avg:.4f}")
    print(f"  F1-Score:  {f1_avg:.4f}")
    print(f"\nSaved Files:")
    print(f"  ✓ best_model_12class_final.keras")
    print(f"  ✓ best_model_12class_final.h5")
    print(f"  ✓ training_curves_12class.png")
    print(f"  ✓ confusion_matrix_12class.png")
    print(f"  ✓ confusion_matrix_normalized_12class.png")
    print(f"  ✓ per_class_metrics_12class.png")
    print(f"  ✓ roc_curves_12class.png")
    print("=" * 80)


if __name__ == "__main__":
    main()

BEST MODEL TRAINING & COMPREHENSIVE ANALYSIS - 12 CLASSES
Best Config: CBAM + 3 Residual Blocks
✓ GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

LOADING DATA...
✓ Train: (64000, 40, 24, 1), Val: (4800, 40, 24, 1), Test: (11200, 40, 24, 1)

Class Distribution:
  Cell: 6004 samples (9.38%)
  Cell-Multi: 4161 samples (6.50%)
  Cracking: 3011 samples (4.70%)
  Diode: 4743 samples (7.41%)
  Diode-Multi: 570 samples (0.89%)
  Hot-spot: 822 samples (1.28%)
  Hot-spot-Multi: 796 samples (1.24%)
  No-anomaly: 31926 samples (49.88%)
  Offline-Module: 2653 samples (4.15%)
  Shadowing: 3382 samples (5.28%)
  Soiling: 675 samples (1.05%)
  Vegetarian: 5257 samples (8.21%)

Class Weights:
  Cell: 0.931
  Cell-Multi: 1.161
  Cracking: 1.409
  Diode: 1.073
  Diode-Multi: 3.825
  Hot-spot: 3.071
  Hot-spot-Multi: 3.131
  No-anomaly: 0.342
  Offline-Module: 1.520
  Shadowing: 1.314
  Soiling: 3.456
  Vegetarian: 1.009

BUILDING BEST MODEL (12-CLASS CLASSIFICATION)


Model: "Best_CBAM_3Residual_12Class"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 40, 24, 1) │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 40, 24,    │        480 │ input[0][0]       │
│                     │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 40, 24,    │        192 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 40, 24,    │          0 │ batch_normalizat… │
│ (Activation)        │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 40, 24,    │          0 │ activation[0][0]  │
│                     │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv2d    │ (None, 40, 24,    │      2,784 │ dropout[0][0]     │
│ (SeparableConv2D)   │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 40, 24,    │        192 │ separable_conv2d… │
│ (BatchNormalizatio… │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 40, 24,    │          0 │ batch_normalizat… │
│ (Activation)        │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 40, 24,    │          0 │ activation_1[0][… │
│                     │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv2d_1  │ (None, 40, 24,    │      2,784 │ dropout_1[0][0]   │
│ (SeparableConv2D)   │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 40, 24,    │        192 │ separable_conv2d… │
│ (BatchNormalizatio… │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 40, 24,    │          0 │ batch_normalizat… │
│ (Activation)        │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 20, 12,    │          0 │ activation_2[0][… │
│ (MaxPooling2D)      │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv2d_2  │ (None, 20, 12,    │      3,960 │ max_pooling2d[0]… │
│ (SeparableConv2D)   │ 72)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 20, 12,    │        288 │ separable_conv2d… │
│ (BatchNormalizatio… │ 72)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 20, 12,    │          0 │ batch_normalizat… │
│ (Activation)        │ 72)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 20, 12,    │          0 │ activation_3[0][… │
│                     │ 72)               │            │                 

 Total params: 183,878 (718.27 KB)

 Trainable params: 180,998 (707.02 KB)

 Non-trainable params: 2,880 (11.25 KB)


✓ Total Parameters: 183,878

TRAINING BEST MODEL
Epoch 1/120
 997/1000 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.3208 - loss: 2.8150
Epoch 1: val_accuracy improved from -inf to 0.55854, saving model to best_model_12class.keras
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 66s 20ms/step - accuracy: 0.3212 - loss: 2.8140 - val_accuracy: 0.5585 - val_loss: 2.2050 - learning_rate: 0.0010
Epoch 2/120
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.5586 - loss: 1.9991
Epoch 2: val_accuracy did not improve from 0.55854
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 16s 16ms/step - accuracy: 0.5586 - loss: 1.9989 - val_accuracy: 0.4671 - val_loss: 2.0020 - learning_rate: 0.0010
Epoch 3/120
 997/1000 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.6010 - loss: 1.6651
Epoch 3: val_accuracy improved from 0.55854 to 0.68563, saving model to best_model_12class.keras
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 21s 16ms/step - accuracy: 0.6010 - loss: 1.6649 - val_accuracy: 0.6856 - val_loss: 1.4310 - learning_rate: 0.0010
Epo

✓ Saved: best_model_12class_final.keras
✓ Saved: best_model_12class_final.h5

EVALUATION

Train Acc: 92.42%  |  Loss: 0.7289
Val Acc:   87.69%  |  Loss: 0.8519
Test Acc:  87.91%  |  Loss: 0.8401

TEST TIME AUGMENTATION (20 iterations)
  Iteration 5/20 complete
  Iteration 10/20 complete
  Iteration 15/20 complete
  Iteration 20/20 complete

✓ TTA Accuracy: 87.91%

DETAILED METRICS

Per-Class Metrics:
Class              Precision    Recall       F1-Score     Support   
----------------------------------------------------------------------
     Cell              0.7498      0.7596      0.7546      1069
  Cell-Multi           0.7125      0.6425      0.6757       702
   Cracking            0.8418      0.8324      0.8371       537
     Diode             0.9522      0.9862      0.9689       869
  Diode-Multi          0.9438      0.9655      0.9545        87
   Hot-spot            0.7616      0.8582      0.8070       134
Hot-spot-Multi         0.7484      0.8623      0.8013       138
  No-ano

/tmp/ipython-input-2989745768.py:582: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  colors = plt.cm.get_cmap('tab20')(np.linspace(0, 1, len(class_names)))


✓ Saved ROC curves: roc_curves_12class.png

🎉 FINAL SUMMARY
Model: CBAM + 3 Residual Blocks (12-Class Classification)
Parameters: 183,878
Training Time: 34.0 minutes

Accuracy:
  Train: 92.42%
  Val:   87.69%
  Test:  87.91%
  TTA:   87.91%

Metrics (Average):
  Precision: 0.8059
  Recall:    0.8411
  F1-Score:  0.8220

Saved Files:
  ✓ best_model_12class_final.keras
  ✓ best_model_12class_final.h5
  ✓ training_curves_12class.png
  ✓ confusion_matrix_12class.png
  ✓ confusion_matrix_normalized_12class.png
  ✓ per_class_metrics_12class.png
  ✓ roc_curves_12class.png
